# Optuna Tuning: Advanced

This notebook covers advanced features of `ModelBasedTuner` with Optuna:

1. **Multi-fidelity pruning** with budget levels (early stopping of bad configs)
2. **BOHB mode** (Bayesian Optimization + HyperBand via Optuna)
3. **Persistent storage** with SQLite for resumable studies
4. **Fidelity slicing** (gradually increase instances and seeds)
5. **Multiple algorithms comparison** (NSGA-II vs MOEA/D vs SPEA2)
6. **Full analysis pipeline** with statistical tests

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from vamos import optimize
from vamos.foundation.quality_indicators import compute_hypervolume
from vamos.engine.tuning import (
    ModelBasedTuner,
    TuningTask,
    Instance,
    EvalContext,
    build_nsgaii_config_space,
    build_moead_config_space,
    build_spea2_config_space,
    config_from_assignment,
    save_history_csv,
    save_history_json,
    vargha_delaney_a12,
    a12_magnitude,
)

plt.style.use("ggplot")
print("Imports OK")

## 1. Multi-Fidelity Pruning with Budget Levels

Instead of always running each config at full budget, we can define **budget levels**.
Optuna evaluates at increasing budgets and **prunes** (stops early) configs that look bad.

This saves time: bad configs are eliminated early, good ones get full budget.

```
Budget levels: [1000, 5000, 25000]

Trial 1: budget=1000 -> score=0.3 (poor) -> PRUNED
Trial 2: budget=1000 -> score=0.7 -> budget=5000 -> score=0.75 -> budget=25000 -> score=0.82
Trial 3: budget=1000 -> score=0.5 -> budget=5000 -> score=0.52 (poor) -> PRUNED
```

In [ ]:
REF_POINT = np.array([1.1, 1.1])
N_VAR = 30

config_space = build_nsgaii_config_space()
param_space = config_space.to_param_space()


def eval_fn_nsgaii(config: dict, ctx: EvalContext) -> float:
    if "pop_size" not in config:
        config["pop_size"] = 100
    cfg = config_from_assignment("nsgaii", config)
    result = optimize(
        ctx.instance.name,
        algorithm="nsgaii",
        algorithm_config=cfg,
        max_evaluations=ctx.budget,
        seed=ctx.seed,
        n_var=ctx.instance.n_var,
        engine="numpy",
    )
    if result.F is None or len(result.F) == 0:
        return 0.0
    return compute_hypervolume(result.F, REF_POINT)


instances = [Instance(name="zdt1", n_var=N_VAR, kwargs={})]

task = TuningTask(
    name="multifidelity_demo",
    param_space=param_space,
    instances=instances,
    seeds=[0, 1, 2],
    budget_per_run=25000,       # max budget
    maximize=True,
    aggregator=lambda s: float(np.mean(s)),
)

print("Task configured with budget_per_run=25000")

In [ ]:
# Multi-fidelity tuner with explicit budget levels
tuner_mf = ModelBasedTuner(
    task=task,
    max_trials=30,
    backend="optuna",
    optuna_sampler="tpe",
    seed=42,
    n_jobs=1,
    budget_levels=[1000, 5000, 25000],  # 3 fidelity levels
)

print("Starting multi-fidelity tuning...")
best_config_mf, history_mf = tuner_mf.run(eval_fn_nsgaii)

# Count pruned vs completed
pruned = sum(1 for t in history_mf if t.details.get("state") == "PRUNED")
completed = sum(1 for t in history_mf if t.details.get("state") == "COMPLETE")

print(f"\nTotal trials: {len(history_mf)}")
print(f"  Completed: {completed}")
print(f"  Pruned:    {pruned} (saved computation!)")
print(f"\nBest HV: {max(t.score for t in history_mf):.6f}")

In [ ]:
# Visualize fidelity traces
fig, ax = plt.subplots(figsize=(10, 5))

for t in history_mf:
    trace = t.details.get("fidelity_trace", [])
    if trace:
        budgets = [step["budget"] for step in trace]
        scores = [step["score"] for step in trace]
        state = t.details.get("state", "UNKNOWN")
        color = "green" if state == "COMPLETE" else "red"
        alpha = 0.8 if state == "COMPLETE" else 0.3
        ax.plot(budgets, scores, "o-", color=color, alpha=alpha, markersize=4)

ax.set_xlabel("Budget (FEs)")
ax.set_ylabel("Hypervolume")
ax.set_title("Multi-Fidelity Traces (green=complete, red=pruned)")
ax.set_xscale("log")
plt.tight_layout()
plt.show()

## 2. BOHB Mode (HyperBand Pruning)

Use `backend="bohb_optuna"` for HyperBand-style successive halving via Optuna. This is more aggressive than median pruning.

In [ ]:
tuner_bohb = ModelBasedTuner(
    task=task,
    max_trials=30,
    backend="bohb_optuna",         # HyperBand pruner
    optuna_sampler="tpe",
    seed=42,
    n_jobs=1,
    budget_levels=[1000, 5000, 25000],
    bohb_reduction_factor=3,       # Keep top 1/3 at each level
)

print("Starting BOHB tuning...")
best_config_bohb, history_bohb = tuner_bohb.run(eval_fn_nsgaii)

pruned_bohb = sum(1 for t in history_bohb if t.details.get("state") == "PRUNED")
print(f"Total: {len(history_bohb)}, Pruned: {pruned_bohb}")
print(f"Best HV: {max(t.score for t in history_bohb):.6f}")

## 3. Persistent Storage (Resumable Studies)

Use `optuna_storage_url` to save the study to an SQLite database. If the notebook crashes, you can resume where you left off.

In [ ]:
DB_PATH = Path("optuna_study.db")

tuner_persistent = ModelBasedTuner(
    task=task,
    max_trials=15,
    backend="optuna",
    optuna_sampler="tpe",
    seed=42,
    n_jobs=1,
    optuna_storage_url=f"sqlite:///{DB_PATH}",  # persistent storage
    optuna_study_name="nsgaii_zdt1_study",        # named study
    optuna_load_if_exists=True,                   # resume if exists
)

print("Running with persistent storage...")
best_config_p, history_p = tuner_persistent.run(eval_fn_nsgaii)

print(f"Trials completed: {len(history_p)}")
print(f"Best HV: {max(t.score for t in history_p):.6f}")
print(f"\nStudy saved to: {DB_PATH.absolute()}")
print("Re-run this cell to add more trials to the same study!")

## 4. Fidelity Slicing (Instance & Seed Subsets)

For large instance sets, low-fidelity evaluations can use **fewer instances and seeds**, scaling up for promising configs.

- `fidelity_min_instance_frac`: fraction of instances at lowest budget
- `fidelity_min_seed_count` / `fidelity_max_seed_count`: seed range

In [ ]:
# Large instance set
many_instances = [
    Instance(name="zdt1", n_var=N_VAR, kwargs={}),
    Instance(name="zdt2", n_var=N_VAR, kwargs={}),
    Instance(name="zdt4", n_var=N_VAR, kwargs={}),
]

task_fidelity = TuningTask(
    name="fidelity_slice_demo",
    param_space=param_space,
    instances=many_instances,
    seeds=[0, 1, 2, 3, 4],         # 5 seeds
    budget_per_run=10000,
    maximize=True,
    aggregator=lambda s: float(np.mean(s)),
)

tuner_fidelity = ModelBasedTuner(
    task=task_fidelity,
    max_trials=20,
    backend="optuna",
    optuna_sampler="tpe",
    seed=42,
    n_jobs=1,
    budget_levels=[2000, 5000, 10000],
    fidelity_min_instance_frac=0.34,   # at lowest budget: ~1/3 of instances
    fidelity_min_seed_count=1,         # at lowest budget: 1 seed
    fidelity_max_seed_count=5,         # at highest budget: all 5 seeds
)

print("Fidelity slicing:")
print("  Budget=2000:  ~1 instance, 1 seed  (fast screening)")
print("  Budget=5000:  ~2 instances, 3 seeds (moderate)")
print("  Budget=10000: 3 instances, 5 seeds  (full evaluation)")

best_config_fs, history_fs = tuner_fidelity.run(eval_fn_nsgaii)
print(f"\nBest HV: {max(t.score for t in history_fs):.6f}")

## 5. Multi-Algorithm Comparison

Tune NSGA-II, MOEA/D, and SPEA2 independently, then compare their best-found configurations.

In [ ]:
ALGORITHMS = {
    "nsgaii": build_nsgaii_config_space,
    "moead": build_moead_config_space,
    "spea2": build_spea2_config_space,
}

task_instances = [Instance(name="zdt1", n_var=N_VAR, kwargs={})]
TUNE_BUDGET = 5000
TUNE_TRIALS = 15

algo_results = {}

for algo_name, build_fn in ALGORITHMS.items():
    print(f"\n=== Tuning {algo_name.upper()} ===")

    cs = build_fn()
    ps = cs.to_param_space()

    def make_eval_fn(alg_name):
        def _eval(config, ctx):
            if "pop_size" not in config:
                config["pop_size"] = 100
            cfg = config_from_assignment(alg_name, config)
            result = optimize(
                ctx.instance.name,
                algorithm=alg_name,
                algorithm_config=cfg,
                max_evaluations=ctx.budget,
                seed=ctx.seed,
                n_var=ctx.instance.n_var,
                engine="numpy",
            )
            if result.F is None or len(result.F) == 0:
                return 0.0
            return compute_hypervolume(result.F, REF_POINT)
        return _eval

    t = TuningTask(
        name=f"tune_{algo_name}",
        param_space=ps,
        instances=task_instances,
        seeds=[0, 1, 2],
        budget_per_run=TUNE_BUDGET,
        maximize=True,
        aggregator=lambda s: float(np.mean(s)),
    )

    tuner = ModelBasedTuner(
        task=t, max_trials=TUNE_TRIALS,
        backend="optuna", optuna_sampler="tpe", seed=42, n_jobs=1,
    )

    best_cfg, hist = tuner.run(make_eval_fn(algo_name))
    scores = [tr.score for tr in hist]
    algo_results[algo_name] = {"best_config": best_cfg, "scores": scores, "best": max(scores)}
    print(f"  Best HV: {max(scores):.6f}")

print("\n--- All algorithms tuned ---")

In [ ]:
# Comparison plot
fig, ax = plt.subplots(figsize=(8, 5))

data = [res["scores"] for res in algo_results.values()]
labels = [f"{name}\n(best={res['best']:.4f})" for name, res in algo_results.items()]

bp = ax.boxplot(data, labels=labels, patch_artist=True)
colors = ["#4C72B0", "#DD8452", "#55A868"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)

ax.set_ylabel("Hypervolume")
ax.set_title("Algorithm Comparison (Optuna-Tuned)")
plt.tight_layout()
plt.show()

## 6. Statistical Analysis with Vargha-Delaney A12

Use the A12 effect size to determine if one algorithm is statistically better than another.

In [ ]:
algo_names = list(algo_results.keys())

print("Pairwise Vargha-Delaney A12 (effect size):")
print("  A12 > 0.5 means row is better than column")
print("  Magnitude: negligible / small / medium / large\n")

for i, a in enumerate(algo_names):
    for j, b in enumerate(algo_names):
        if i >= j:
            continue
        scores_a = np.array(algo_results[a]["scores"])
        scores_b = np.array(algo_results[b]["scores"])
        a12 = vargha_delaney_a12(scores_a, scores_b)
        magnitude = a12_magnitude(a12)
        winner = a if a12 > 0.5 else b
        print(f"  {a.upper()} vs {b.upper()}: A12={a12:.3f} ({magnitude}) -> {winner.upper()} tends to be better")

## 7. Export Full Results

In [ ]:
# Summary table
summary = pd.DataFrame({
    algo: {
        "best_hv": res["best"],
        "mean_hv": np.mean(res["scores"]),
        "std_hv": np.std(res["scores"]),
        "n_trials": len(res["scores"]),
    }
    for algo, res in algo_results.items()
}).T

print("\nSummary:")
print(summary.to_string())

# Save best configs
print("\nBest configurations:")
for algo, res in algo_results.items():
    print(f"\n  {algo.upper()}:")
    for k, v in res["best_config"].items():
        print(f"    {k}: {v}")

## Summary

| Feature | Parameter | Description |
|---|---|---|
| Multi-fidelity | `budget_levels=[1000, 5000, 25000]` | Evaluate at increasing budgets, prune bad configs early |
| BOHB mode | `backend="bohb_optuna"` | HyperBand-style successive halving |
| Persistent storage | `optuna_storage_url="sqlite:///study.db"` | Resumable studies across sessions |
| Fidelity slicing | `fidelity_min_instance_frac`, `fidelity_*_seed_count` | Use fewer instances/seeds at low budgets |
| Statistical tests | `vargha_delaney_a12(a, b)` | Effect size for algorithm comparison |